# Week 3 — First Agents with LangGraph

Three agents built with LangGraph's prebuilt `create_react_agent`, covering this week's learning objectives: **agent = LLM + tools + memory + control flow**.

1. **Calculator agent** — one tool, arithmetic only
2. **Web Search agent** — one tool, live web search (DuckDuckGo, no API key needed)
3. **Multi-tool agent with conversation memory** — both tools combined, plus a LangGraph checkpointer so the agent remembers earlier turns in the same conversation

Same Groq setup as Week 2 (no OpenAI billing dependency).

## 1. Setup — LLM client

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(dotenv_path="/Users/revathy_santhosh/rag-project/.env")

# Groq's API is OpenAI-compatible, so we point LangChain's standard ChatOpenAI
# client at Groq's endpoint instead of using the separate langchain-groq package
# (which conflicts with the newer groq SDK version already in this project).
llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

print("LLM ready.")

LLM ready.


## 2. Agent 1 — Calculator

A single tool that safely evaluates arithmetic expressions (using Python's `ast` module, not a raw `eval`, so the agent can't execute arbitrary code — just math).

In [2]:
import ast
import operator
from langchain_core.tools import tool

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")

@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression. Supports +, -, *, /, ** and parentheses.
    Example input: '23 * 47 + 19'
    """
    try:
        tree = ast.parse(expression, mode="eval").body
        result = _safe_eval(tree)
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"

print(calculator.invoke({"expression": "23 * 47 + 19"}))

1100


In [3]:
from langgraph.prebuilt import create_react_agent

calculator_agent = create_react_agent(llm, tools=[calculator])

response = calculator_agent.invoke(
    {"messages": [("user", "What is 23 times 47, plus 19? Use the calculator tool.")]}
)
print(response["messages"][-1].content)

/Users/revathy_santhosh/rag-project/.venv/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
/var/folders/9f/dvpbk9t127702zbf5t3h60ch0000gn/T/ipykernel_2620/254534406.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  calculator_agent = create_react_agent(llm, tools=[calculator])


The calculation evaluates to **1,100**.


## 3. Agent 2 — Web Search

Uses DuckDuckGo search (via `langchain_community`'s `DuckDuckGoSearchRun`) — free, no API key or billing setup needed, unlike Tavily/SerpAPI.

In [4]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

search_agent = create_react_agent(llm, tools=[search_tool])

response = search_agent.invoke(
    {"messages": [("user", "Search the web and tell me, in one sentence, what LangGraph is used for.")]}
)
print(response["messages"][-1].content)

/var/folders/9f/dvpbk9t127702zbf5t3h60ch0000gn/T/ipykernel_2620/1908676866.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  search_agent = create_react_agent(llm, tools=[search_tool])
/Users/revathy_santhosh/rag-project/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/Users/revathy_santhosh/rag-project/.venv/lib/python3.12/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


LangGraph is an open‑source framework from LangChain that orchestrates, executes, and manages stateful AI agent workflows—offering durable execution, streaming, memory, and human‑in‑the‑loop capabilities.


## 4. Agent 3 — Multi-tool agent with conversation memory

Combines both tools, and adds a `MemorySaver` checkpointer so the agent remembers prior turns *within the same conversation thread* (`thread_id`). This is LangGraph's short-term/conversation memory, built on the same state-graph + checkpointer concept from this week's learning objectives.

In [5]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

multi_tool_agent = create_react_agent(
    llm,
    tools=[calculator, search_tool],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "demo-conversation-1"}}

response1 = multi_tool_agent.invoke(
    {"messages": [("user", "Hi, I'm Revathy. What's 150 times 3?")]},
    config=config,
)
print("Turn 1:", response1["messages"][-1].content)

/var/folders/9f/dvpbk9t127702zbf5t3h60ch0000gn/T/ipykernel_2620/989175349.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  multi_tool_agent = create_react_agent(


Turn 1: Hi Revathy!  
\(150 \times 3 = 450\).


In [6]:
# Follow-up turn, same thread_id -- the agent should remember the name and the earlier calculation
response2 = multi_tool_agent.invoke(
    {"messages": [("user", "What's my name, and what did I just ask you to calculate?")]},
    config=config,
)
print("Turn 2:", response2["messages"][-1].content)

Turn 2: You’re Revathy, and you just asked me to calculate \(150 \times 3\).


### Proving the memory is actually scoped per conversation

A **different** `thread_id` should have no memory of the conversation above -- this confirms the checkpointer memory is per-thread, not global.

In [7]:
new_config = {"configurable": {"thread_id": "demo-conversation-2"}}

response3 = multi_tool_agent.invoke(
    {"messages": [("user", "What's my name?")]},
    config=new_config,
)
print("New thread (should NOT know the name):", response3["messages"][-1].content)

New thread (should NOT know the name): I don’t have that information. If you let me know, I’ll remember it for our conversation.


## Recap

- **Agent 1 (Calculator)**: `create_react_agent` + one custom tool (`@tool`-decorated, AST-based safe arithmetic evaluator)
- **Agent 2 (Web Search)**: `create_react_agent` + `DuckDuckGoSearchRun`, a prebuilt LangChain tool, no API key required
- **Agent 3 (Multi-tool + memory)**: both tools combined, plus `MemorySaver` as a checkpointer, demonstrating that conversation memory is scoped to a `thread_id` -- the same agent remembers context within one thread and starts fresh in another

This covers the week's learning objectives: agent = LLM + tools + memory + control flow, using LangGraph's prebuilt ReAct agent pattern, with conversation memory via a checkpointer.